# 00 — Data Model Overview

This is the map of the warehouse behind the demo. It is a **local DuckDB file**
(`data/warehouse/retail.duckdb`) generated entirely by `src/retail_synth/`, structured
in three medallion layers so it can be lifted onto a Microsoft Fabric Lakehouse later
with minimal change:

- **bronze** — near-raw tables loaded 1:1 from the generated parquet in `data/raw/`
- **silver** — typed, conformed tables (same shape as bronze here, since the generator
  already produces clean data — a couple of tables get light cleaning via
  `sql/silver/*.sql`)
- **gold** — decision-ready marts: pre-aggregated demand, inventory-imbalance signals,
  a returns scorecard, supply-risk exposure, a customer-cohort scorecard, and the
  rules-derived `decision_queue` that powers notebook 08's "5 decisions" screen.

The dataset tells one continuous story: a premium apparel retailer approaching peak
winter season, with six deliberately engineered situations layered into otherwise
ordinary multi-year history (see `ai_native_retail_strategy.md` for the brief this
was built from). Notebooks 02–07 investigate each situation; notebook 08 assembles
the planner's Monday-morning workspace.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Tables by layer

In [2]:
for schema in ("bronze", "silver", "gold"):
    tabs = con.execute(f"""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema='{schema}' ORDER BY 1
    """).fetchall()
    print(f"\n{schema} ({len(tabs)} tables)")
    for (t,) in tabs:
        n = con.execute(f"SELECT COUNT(*) FROM {schema}.{t}").fetchone()[0]
        print(f"  {t:<32} {n:>12,} rows")


bronze (21 tables)
  dim_campaign                               45 rows
  dim_customer                          500,000 rows
  dim_dc                                      6 rows
  dim_fx_rate                               936 rows
  dim_region                                 18 rows
  dim_return_reason                           8 rows
  dim_sku                                21,619 rows
  dim_store                                 280 rows
  dim_style                               1,200 rows
  dim_supplier                               60 rows
  dim_week                                  156 rows
  fact_campaign_exposure                385,925 rows
  fact_digital_engagement               621,913 rows
  fact_inventory_position            12,035,441 rows
  fact_purchase_order_line                4,743 rows
  fact_returns_line                   1,007,239 rows
  fact_sales_line                     5,514,181 rows
  fact_shipment_event                    18,957 rows
  fact_weather_actual     

## Entity model (short version)

**Dimensions:** `dim_week` (fiscal calendar), `dim_region` (18 regions, doubles as the
weather grain), `dim_dc` (6 distribution centers), `dim_store` (280 stores),
`dim_style` / `dim_sku` (1,200 styles → ~25k SKUs), `dim_customer` (500k),
`dim_supplier` (60), `dim_campaign` (45), `dim_return_reason`, `dim_fx_rate`.

**Bridge:** `store_sku_assortment` — which SKUs are listed where (physical store or
one virtual `ECOM-<region>` location per region) and for how long. Nothing gets
generated for a location-SKU pair outside its listing window — this is what keeps
the fact tables sparse instead of a dense cross join.

**Facts:** `fact_sales_line`, `fact_inventory_position` (the two big ones, produced
together by the core demand/supply engine), `fact_returns_line`,
`fact_purchase_order_line` + `fact_shipment_event`, `fact_weather_actual` +
`fact_weather_forecast_snapshot`, `fact_campaign_exposure`, `fact_digital_engagement`.

**Gold marts:** `weekly_demand_style_region`, `weekly_demand_store_sku`,
`inventory_imbalance_signals`, `returns_scorecard`, `supply_risk_exposure`,
`customer_cohort_scorecard`, `decision_queue`, `approved_actions`.

## Sample rows

In [3]:
con.execute("SELECT * FROM silver.dim_store LIMIT 5").df()

,store_id,store_name,region_code,country,city,climate_zone,store_tier,square_meters,lat,long,open_date,primary_dc_id,currency_code
0,ST-TOR-01,Toronto Eaton Centre,ONT,CA,Toronto,cold,Flagship,1413,57.6,-103.3,2008-12-12,DC-NAEAST,CAD
1,ST-TOR-02,Toronto Yorkdale,ONT,CA,Toronto,cold,A,470,52.3,-99.0,2021-02-10,DC-NAEAST,CAD
2,ST-VAN-01,Vancouver Pacific Centre,WCA,CA,Vancouver,cold,Flagship,905,51.4,-106.5,2019-12-05,DC-NAWEST,CAD
3,ST-VAN-02,Vancouver Oakridge,WCA,CA,Vancouver,cold,A,470,56.5,-115.9,2008-05-30,DC-NAWEST,CAD
4,ST-NYC-01,New York Fifth Avenue,USNE,US,New York,cold,Flagship,1450,43.3,-96.3,2009-10-22,DC-NAEAST,USD


In [4]:
con.execute("""
    SELECT decision_id, decision_type, headline, metric_value
    FROM gold.decision_queue ORDER BY decision_type
""").df()

,decision_id,decision_type,headline,metric_value
0,DEC-0299,Cohort Opportunity,Customers acquired through the Milan campaign ...,2.6
1,DEC-0121,Overstock,Chelsea Boot 0100 -- USMW inventory is project...,686.3
2,DEC-0092,Overstock,Bomber 0531 -- NOR inventory is projected to e...,750.4
3,DEC-0003,Overstock,Bag 0504 -- BLX inventory is projected to exce...,567.9
4,DEC-0123,Overstock,Derby 1119 -- USMW inventory is projected to e...,808.6
...,...,...,...,...
294,DEC-0118,Stockout,Gloves 0440 -- USMW is expected to stock out w...,9.0
295,DEC-0177,Stockout,Loafer 0685 -- WCA is expected to stock out wi...,10.3
296,DEC-0120,Stockout,Aurora Bomber -- USMW is expected to stock out...,0.7
297,DEC-0043,Stockout,Gloves 0878 -- FRA is expected to stock out wi...,11.8


## Fabric portability note

`sql/silver/*.sql` and `sql/gold/*.sql` are plain SQL against `bronze.*` / `silver.*`
tables — moving this to Fabric later means pointing the same queries at a Lakehouse
SQL endpoint (bronze/silver as Lakehouse tables, gold as Warehouse or a semantic
model) instead of a local DuckDB file. The generation code in `src/retail_synth/`
stays local either way — only the destination of `load_bronze.py` changes.